# Task A: Nigerian Petrol Price Forecasting — a Feature Engineering Ladder

**Primary task of the v3 modelling layer.** Predicts `price_ngn` for a given
Nigerian state **one month ahead**, using `fact_fuel_price_monthly` — the
platform's complete, reconciled petrol price panel: **1,147 rows, 37 states,
31 months (November 2023 – May 2026)**, verified against NBS's own published
national means to within ₦0.00 (`outputs/quality/quality_report.md`). This is
the paper's central contribution: a small, real, completely-covered dataset,
which is exactly where data engineering choices are most visible in the
result.

**The model class is fixed across every rung of this ladder: a single
`LGBMRegressor` (LightGBM gradient-boosted trees), every time**, except that
rung V4 fits it twice — once per candidate encoding — because encoding is
the thing V4 compares. Fixing the model class is the whole point of the
exercise: only the data changes between rungs, so any change in error is
attributable to the data engineering decision, not to a different algorithm.

**This notebook runs the ladder TWICE, with two different fixed
hyperparameter sets, reported side by side throughout:**

- **original (untuned, 300 trees)** — the first version built: 300
  estimators, 31 leaves, no L1/L2, picked without any search.
- **capacity-controlled (CV-selected on V0)** — hyperparameters chosen
  **once**, before any rung is fit, by expanding-window cross-validation
  confined to the training period, run only on V0's own baseline feature
  (`price_ngn`), optimising MAE (§5 below). Those hyperparameters are then
  frozen and reused, unchanged, at every rung — the same "only the data
  changes" discipline as the original ladder, just applied to a model
  capacity actually suited to a 925-row training panel.

**And every rung of both variants is run FIVE times, with error bars.** A
ladder that reports one number per rung cannot support a claim that one rung
beat another, because it never measured how much a rung's number moves when
nothing meaningful changes. Every rung below is therefore run once per seed
in `splits.REPEAT_SEEDS` (**1, 2, 3, 4, 5**), varying the LightGBM
`random_state` — the model's bagging and feature-sampling randomness — while
holding the data and every hyperparameter fixed. What is reported is the
**mean across those five repeats, its standard deviation, and its range**,
and, for every comparison that matters, whether the five repeats **agree on
the direction** of the difference (§9). A difference that does not hold its
sign across all five repeats is not treated as established, however good its
mean looks.

**Two reference predictors are reported alongside the rungs (§6)**, because a
ladder shows relative movement and never answers "is this error any good?":
the training mean, and a random walk (next month's price = this month's).

Error is reported two ways: **Mean Absolute Error in naira** and **Mean
Absolute Percentage Error**, since the price level rose 146% over the series.

In [1]:
import sys, time, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=UserWarning)

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import settings
from src.viz import style as vizstyle
from src.modelling.splits import (
    nigeria_time_split, assert_split_is_time_ordered, REPEAT_SEEDS,
    NIGERIA_TRAIN_END_KEY, NIGERIA_TEST_START_KEY, NIGERIA_TEST_END_KEY,
)
from src.modelling.features import add_leaky_state_avg_nigeria, add_pit_state_avg_nigeria
from src.modelling.encoders import one_hot_encode, target_encode_pit
from src.modelling.ladder import (
    run_rung, score_predictions, LGBM_PARAMS,
    VARIANT_ORIGINAL, VARIANT_CAPACITY_CONTROLLED, VARIANT_REFERENCE,
)
from src.modelling.tuning import expanding_window_folds, select_hyperparameters_cv
from src.modelling.baselines import predict_train_mean, predict_last_value_carried_forward
from src.modelling.repeats import (
    Pair, adjacent_pairs, aggregate_repeats, paired_comparisons, best_rung_by_mean_mae,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

NOTEBOOK_STARTED = time.perf_counter()

con = duckdb.connect(str(settings.WAREHOUSE_DB), read_only=True)
print("warehouse:", settings.WAREHOUSE_DB)
print("repeat seeds:", REPEAT_SEEDS)
print()
print("ORIGINAL hyperparameters (first ladder variant, every rung):")
LGBM_PARAMS

warehouse: /app/data/warehouse.duckdb
repeat seeds: (1, 2, 3, 4, 5)

ORIGINAL hyperparameters (first ladder variant, every rung):


{'n_estimators': 300,
 'learning_rate': 0.05,
 'num_leaves': 31,
 'max_depth': -1,
 'min_child_samples': 20,
 'subsample': 0.8,
 'subsample_freq': 1,
 'colsample_bytree': 0.8,
 'random_state': 796,
 'n_jobs': 4,
 'verbosity': -1}

## 1. Data: the complete, reconciled fuel price panel

`fact_fuel_price_monthly` is joined to `dim_geography` (for each state's
`region_group`, its geopolitical zone) and `dim_date` (for `month_number`,
used in rung V2). Both joins are read-only lookups against the same gold
layer the paper's A12 analysis (the centrepiece Nigerian analysis) already
uses.

In [2]:
raw = con.execute('''
    SELECT f.month_key,
           f.geo_key,
           g.geo_name       AS state,
           g.region_group   AS region_group,
           d.month_number   AS month_of_year,
           f.price_ngn
    FROM fact_fuel_price_monthly f
    JOIN dim_geography g ON g.geo_key = f.geo_key AND g.is_current
    JOIN dim_date d      ON d.date_key = f.month_key
    ORDER BY g.geo_name, f.month_key
''').df()

print(f"{len(raw):,} rows, {raw['state'].nunique()} states, "
      f"{raw['month_key'].nunique()} months "
      f"({raw['month_key'].min()} .. {raw['month_key'].max()})")
assert len(raw) == 1147, "expected the full 37 x 31 rectangular panel"
raw.head()

1,147 rows, 37 states, 31 months (20231101 .. 20260501)


,month_key,geo_key,state,region_group,month_of_year,price_ngn
0,20231101,119,Abia,South East,11,654.3571
1,20231201,119,Abia,South East,12,690.5263
2,20240101,119,Abia,South East,1,687.5000
3,20240201,119,Abia,South East,2,688.6200
4,20240301,119,Abia,South East,3,710.8333


## 2. Building the one-month-ahead supervised frame

Each row of the raw panel is `(state, month)`. The prediction target is that
same state's price in the **next** calendar month, built with a per-state
`shift(-1)` after sorting by `(state, month_key)` — never a random shuffle.

**An honest wrinkle, stated here rather than glossed over:** the panel's
final month, 2026-05, has no "next month" to supply a target for, so every
state's 2026-05 row is dropped before modelling (37 rows). The 31-month raw
panel therefore yields a **30-feature-month modelling frame**. The v3 prompt
describes the test window as "2025-12 through 2026-05 (6 months)"; applied to
each row's *own* `month_key` (as `src/modelling/splits.py` does, consistently
with how the training boundary is stated), the test side of the *modelling*
frame covers **5** feature months (2025-12 to 2026-04), predicting target
months 2026-01 through 2026-05 — the last month the panel can possibly
predict. This is a direct, unavoidable consequence of framing the problem as
one-month-ahead lag prediction on a fixed-length panel, not a deviation from
the split boundary itself, which is applied exactly as specified.

In [3]:
raw = raw.sort_values(["state", "month_key"]).reset_index(drop=True)
raw["target_price_next_month"] = raw.groupby("state")["price_ngn"].shift(-1)

modeling = raw.dropna(subset=["target_price_next_month"]).copy()
modeling["region_group"] = modeling["region_group"].astype("category")

print(f"raw panel:      {len(raw):,} rows ({raw['month_key'].nunique()} months)")
print(f"modelling frame: {len(modeling):,} rows ({modeling['month_key'].nunique()} feature months) "
      f"across {modeling['state'].nunique()} states")
modeling[["state", "month_key", "price_ngn", "target_price_next_month", "region_group", "month_of_year"]].head()

raw panel:      1,147 rows (31 months)
modelling frame: 1,110 rows (30 feature months) across 37 states


,state,month_key,price_ngn,target_price_next_month,region_group,month_of_year
0,Abia,20231101,654.3571,690.5263,South East,11
1,Abia,20231201,690.5263,687.5000,South East,12
2,Abia,20240101,687.5000,688.6200,South East,1
3,Abia,20240201,688.6200,710.8333,South East,2
4,Abia,20240301,710.8333,706.0000,South East,3


## 3. Train / test split — by time, never shuffled

`src/modelling/splits.nigeria_time_split` splits on each row's own
`month_key`: **train ≤ 2025-11, test 2025-12 .. 2026-05** — the exact
boundary stated in the v3 prompt §3.2. `assert_split_is_time_ordered` is a
live, re-executable check (the same assertion `tests/test_time_split.py`
runs against this table) that the maximum training `month_key` is strictly
less than the minimum test `month_key`; it raises if that is ever untrue.

**Task A's data does not vary between repeats.** There is no sampling here —
the panel is complete and every rung uses all of it — so the five repeats
vary only the model's own randomness. Task B, which does sample, varies the
sample as well.

In [4]:
split = nigeria_time_split(modeling)
assert_split_is_time_ordered(split, time_col="month_key")
print(split.boundary_description)
print()
print("train feature months:", sorted(split.train["month_key"].unique()))
print("test  feature months:", sorted(split.test["month_key"].unique()))

train: month_key <= 20251101 (25 months, 925 rows); test: 20251201 <= month_key <= 20260501 (5 months, 185 rows)

train feature months: [np.int32(20231101), np.int32(20231201), np.int32(20240101), np.int32(20240201), np.int32(20240301), np.int32(20240401), np.int32(20240501), np.int32(20240601), np.int32(20240701), np.int32(20240801), np.int32(20240901), np.int32(20241001), np.int32(20241101), np.int32(20241201), np.int32(20250101), np.int32(20250201), np.int32(20250301), np.int32(20250401), np.int32(20250501), np.int32(20250601), np.int32(20250701), np.int32(20250801), np.int32(20250901), np.int32(20251001), np.int32(20251101)]
test  feature months: [np.int32(20251201), np.int32(20260101), np.int32(20260201), np.int32(20260301), np.int32(20260401)]


## 4. Attaching every rung's features once, up front

Historical-aggregate and target-encoded features (V3a/V3b/V4) are attached to
the **full** `modeling` frame (train and test rows together) before the
split's columns are selected per rung. This is deliberate, not a shortcut:
the leaky builder's entire failure mode requires the test rows to be
physically present when it computes "this state's average price," and the
point-in-time builder is safe to compute the same way because, for every
test row, that row's "prior" window is by construction entirely inside the
training period. See `src/modelling/features.py` and
`src/modelling/encoders.py` for the two clearly separate code paths behind
each pair of columns added below.

In [5]:
modeling = add_leaky_state_avg_nigeria(modeling, state_col="state")
modeling = add_pit_state_avg_nigeria(modeling, state_col="state")
modeling = target_encode_pit(
    modeling, cat_col="state", target_col="target_price_next_month",
    time_col="month_key", smoothing=10.0, out_col="state_target_enc",
)

split = nigeria_time_split(modeling)  # rebuild after attaching new columns
assert_split_is_time_ordered(split, time_col="month_key")

TARGET = "target_price_next_month"
y_train, y_test = split.train[TARGET], split.test[TARGET]

modeling[[
    "state", "month_key", "price_ngn", "state_avg_price_leaky",
    "state_avg_price_pit", "state_target_enc",
]].head(8)

,state,month_key,price_ngn,state_avg_price_leaky,state_avg_price_pit,state_target_enc
0,Abia,20231101,654.3571,1013.79235,NaN,1029.250510
1,Abia,20231201,690.5263,1013.79235,654.357100,673.555307
2,Abia,20240101,687.5000,1013.79235,672.441700,673.233343
3,Abia,20240201,688.6200,1013.79235,677.461133,676.798241
4,Abia,20240301,710.8333,1013.79235,680.250850,683.446128
5,Abia,20240401,706.0000,1013.79235,686.367340,687.905428
6,Abia,20240501,770.0000,1013.79235,689.639450,702.005802
7,Abia,20240601,794.9091,1013.79235,701.119529,711.866580


## 5. Once-only hyperparameter selection: a model capacity that fits 925 rows

**Why this step exists.** The original variant uses a fixed 300-tree,
31-leaf `LGBMRegressor` with no L1/L2 regularisation at every rung. On a
925-row training panel, a model with that much capacity can fit essentially
any feature set it is given, including noise — which means a ladder built
on it cannot distinguish "this feature helped" from "this model overfit a
little differently." That is a mismatch between model capacity and sample
size, not a flaw in the ladder's *design*.

**The fix, and the rule that keeps it from becoming a second confound.**
Hyperparameters are selected **once**, before any rung is fit, using
**only V0's own baseline feature (`price_ngn`) and V0's own data** — never a
feature engineered by a later rung, and never any test-period row. The
search uses **expanding-window cross-validation** confined entirely to the
25 training months: fold *k* trains on every month up to a growing cutoff
and validates on the next block of months strictly after it, so every fold
is itself a smaller, honest time-based split
(`src/modelling/tuning.expanding_window_folds`,
`tests/test_tuning.py` proves the folds never let validation precede or
overlap training). The search optimises **MAE**, sweeps `num_leaves`,
`learning_rate`, `min_child_samples`, `reg_alpha` (L1) and `reg_lambda` (L2)
over a curated set of candidates ranging from very small capacity to close
to the original default, and selects `n_estimators` per candidate via early
stopping on each fold's validation MAE. The winning candidate's
`n_estimators` is frozen as the mean of its best iterations on the **two
largest-training-window folds** — not a median across all folds, whose
smaller early windows would fix a tree count sized for less data than the
ladder's own 25-month fit ever uses (`src/modelling/tuning.py`).

**This selection runs once, not once per repeat.** Re-selecting per seed
would make the capacity-controlled variant a different model on every
repeat, and the spread across repeats would then confound model randomness
with hyperparameter churn. The five repeats vary `random_state` alone.

**The winning hyperparameters are reused, unchanged, at every rung below —
never re-tuned per rung.** That is what keeps "only the data changes" true
for the capacity-controlled ladder exactly as it was for the original one.

In [6]:
FEATURES_V0 = ["price_ngn"]

train_months = sorted(split.train["month_key"].unique())
cv_folds = expanding_window_folds(train_months, min_train_months=15, val_block=2)
print(f"{len(cv_folds)} expanding-window folds over the {len(train_months)} training months:")
for f in cv_folds:
    print(f"  train <= {f.train_months[-1]} ({len(f.train_months)} months)  "
          f"-> validate {f.val_months}")

tuning = select_hyperparameters_cv(
    split.train, feature_cols=FEATURES_V0, target_col=TARGET,
    month_col="month_key", folds=cv_folds,
)
print(f"\nCV search: {len(tuning.cv_table)} candidates x up to {tuning.n_folds} folds "
      f"in {tuning.seconds:.1f}s")
tuning.cv_table

5 expanding-window folds over the 25 training months:
  train <= 20250101 (15 months)  -> validate (np.int32(20250201), np.int32(20250301))
  train <= 20250301 (17 months)  -> validate (np.int32(20250401), np.int32(20250501))
  train <= 20250501 (19 months)  -> validate (np.int32(20250601), np.int32(20250701))
  train <= 20250701 (21 months)  -> validate (np.int32(20250801), np.int32(20250901))
  train <= 20250901 (23 months)  -> validate (np.int32(20251001), np.int32(20251101))



CV search: 8 candidates x up to 5 folds in 1.4s


,num_leaves,learning_rate,min_child_samples,reg_alpha,reg_lambda,mean_cv_mae,n_folds_used,median_best_iteration,largest_fold_best_iteration
0,7,0.10,5,0.0,0.0,39.475435,5,10,12
1,7,0.03,5,0.0,0.0,39.895027,5,34,42
2,15,0.05,10,0.0,0.0,41.530196,5,23,26
3,7,0.10,20,1.0,1.0,42.007247,5,9,12
4,15,0.05,20,1.0,1.0,42.800057,5,22,26
5,31,0.05,20,1.0,1.0,42.862893,5,21,26
6,15,0.10,30,0.5,0.5,44.153061,5,12,13
7,31,0.10,50,1.0,1.0,45.735052,5,11,12


In [7]:
CAPACITY_CONTROLLED_PARAMS = {
    **tuning.best_params,
    "n_estimators": tuning.frozen_n_estimators,
    "max_depth": -1,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "random_state": 796,   # overridden per repeat by run_rung(seed=...)
    "n_jobs": 4,
    "verbosity": -1,
}
print("Selected once, on V0 alone, then frozen for every rung and every repeat below:")
CAPACITY_CONTROLLED_PARAMS

Selected once, on V0 alone, then frozen for every rung and every repeat below:


{'num_leaves': 7,
 'learning_rate': 0.1,
 'min_child_samples': 5,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'n_estimators': 12,
 'max_depth': -1,
 'subsample': 0.8,
 'subsample_freq': 1,
 'colsample_bytree': 0.8,
 'random_state': 796,
 'n_jobs': 4,
 'verbosity': -1}

## 6. Reference predictors: is any of this error actually good?

The ladder measures each rung against the rung below it. That says nothing
about whether the whole ladder sits anywhere useful. Two reference
predictors, fitting no model at all, supply that anchor
(`src/modelling/baselines.py`):

- **REF_mean** — predict the training set's mean target price for every test
  row. This is the floor: a rung that cannot beat it has learned nothing
  from its features whatsoever.
- **REF_heuristic** — a **random walk**: next month's price equals this
  month's, carried forward unchanged. This is the standard naive benchmark
  for a price series, and on Task A it is a genuinely hard bar, because
  V0 is a gradient booster handed that same single column and asked to
  learn a mapping from it. Any gap between V0 and this line is the model's
  learned adjustment on top of pure persistence — which, on a series still
  trending upward through the test window, is not guaranteed to be an
  improvement.

Both are deterministic here: no model, no sampling, so they return the
identical value on every repeat and their standard deviation is
**structurally zero**, not a measured stability. They are labelled with
their own variant (`reference (no model fitted)`) so they can never be
mistaken for a rung or drawn into a rung-to-rung comparison.

No published literature figure is quoted as a comparison point anywhere in
this layer. A published Nigerian fuel-price error is not comparable to this
one unless the window, the states, the target definition and the split all
match, and they do not — so these internal references do that job instead.

In [8]:
def reference_results(seed):
    '''The two no-model reference predictors, scored exactly like a rung.'''
    ref_mean = score_predictions(
        "REF_mean", "reference: predict the training mean price for every test row",
        y_test, predict_train_mean(y_train, len(y_test)), n_train=len(y_train),
        notes="no model, no features -- the floor any rung must beat to have learned anything",
        variant=VARIANT_REFERENCE, seed=seed,
    )
    ref_walk = score_predictions(
        "REF_heuristic", "reference: random walk -- next month's price = this month's price",
        y_test, predict_last_value_carried_forward(split.test, "price_ngn"), n_train=len(y_train),
        notes="the standard naive benchmark for a price series; deterministic, so its spread "
              "across repeats is structurally zero rather than measured",
        variant=VARIANT_REFERENCE, seed=seed,
    )
    return [ref_mean, ref_walk]

_preview = reference_results(REPEAT_SEEDS[0])
for r in _preview:
    print(f"{r.rung:14s} MAE=\u20a6{r.mae:,.2f}   MAPE={r.mape:.2f}%")

REF_mean       MAE=₦327.34   MAPE=22.54%
REF_heuristic  MAE=₦129.30   MAPE=9.57%


## 7. The rungs

Each rung below declares what it adds to the feature set. Nothing is fitted
yet: the rungs are declared here, in order, and then §8 runs every one of
them under both hyperparameter variants, five times each. Collecting the
declarations first and executing them in one place is what guarantees every
repeat runs the identical ladder — a rung cannot accidentally differ between
seeds, because there is only one definition of it.

In [9]:
RUNG_SPECS = []

def add_rung(rung, description, notes, feature_cols=None, kind="columns"):
    RUNG_SPECS.append({"rung": rung, "description": description, "notes": notes,
                       "feature_cols": feature_cols, "kind": kind})
    return feature_cols

### V0 — naive baseline

The single feature a practitioner reaches for with zero data-engineering
effort: this month's own raw price, predicting next month's. No other
column, no cleaning beyond what loading the table already requires.

In [10]:
add_rung("V0", "naive baseline: current month's raw price only",
         "no data engineering: the one feature a practitioner reaches for with zero effort",
         FEATURES_V0)

['price_ngn']

### V1 — quality-gated

Identical features to V0, this time read through the platform's fully
validated path (the `dim_geography`/`dim_date` join above, which fails
silently to `NULL` if a `geo_key` or `month_key` cannot be resolved — it did
not, for any of these 1,147 rows). `fact_fuel_price_monthly` had **zero**
rejected rows at the quality gate (`outputs/quality/quality_report.md`, rule
Q002/Q015: 1,147 checked, 0 failed). **V0 and V1 therefore read the
identical validated rows**, so an identical metric here is the correct,
honest result of this rung — not a failed experiment, under either
hyperparameter variant. With repeats, this rung becomes a useful internal
control: V0 and V1 are the same fit on the same data with the same seed, so
their paired difference must be exactly zero at every seed. Anything else
would indicate a bug in the harness.

In [11]:
FEATURES_V1 = FEATURES_V0
add_rung("V1", "quality-gated: identical features, sourced from fact_fuel_price_monthly "
               "after the quality gate (0 of 1,147 rows rejected)",
         "fact_fuel_price_monthly had zero rejected rows (quality_report.md); V0 and V1 "
         "read identical validated rows, so an unchanged metric here is the honest result",
         FEATURES_V1)

['price_ngn']

### V2 — dimensional features

Adds two attributes pulled through the *conformed dimensions*, not the bare
fact row: `region_group` (a state's geopolitical zone, from `dim_geography`)
and `month_of_year` (from `dim_date`, capturing seasonality — e.g. dry-season
transport-cost effects). This tests whether attaching descriptive dimensional
context, the whole premise of a Kimball-style star schema, buys anything a
one-column fact table cannot.

In [12]:
FEATURES_V2 = FEATURES_V1 + ["region_group", "month_of_year"]
add_rung("V2", "+ dimensional features: region_group (dim_geography) and month_of_year (dim_date)",
         "tests whether conformed-dimension context improves on bare fact columns",
         FEATURES_V2)

['price_ngn', 'region_group', 'month_of_year']

### V3a — naive historical aggregate (⚠️ LEAKY, DO NOT TRUST THIS NUMBER)

Adds **"this state's average price"**, computed by
`add_leaky_state_avg_nigeria` over the **entire** dataset — including months
after the row being featurised. A 2023-11 row receives a value partly
computed from 2026-05, a month that, from that row's vantage point, had not
happened yet. Reported for comparison only; **never to be read as a real
capability of the model, under either hyperparameter variant.**

In [13]:
FEATURES_V3A = FEATURES_V2 + ["state_avg_price_leaky"]
add_rung("V3a", "LEAKY, DO NOT TRUST THIS NUMBER: + state_avg_price_leaky "
                "(this state's average price over ALL months, including future ones)",
         "LEAKY, DO NOT TRUST THIS NUMBER -- computed with future months present; "
         "reported only to be compared against V3b",
         FEATURES_V3A)

['price_ngn', 'region_group', 'month_of_year', 'state_avg_price_leaky']

### V3b — point-in-time-correct historical aggregate

The same idea, done correctly. `add_pit_state_avg_nigeria` computes an
**expanding mean per state, using only months strictly before** the row's
own `month_key`. A state's first observed month has no prior data at all and
is `NaN` by construction (LightGBM handles missing values natively; nothing
here is imputed).

In [14]:
FEATURES_V3B = FEATURES_V2 + ["state_avg_price_pit"]
add_rung("V3b", "+ state_avg_price_pit (this state's average price, strictly-prior months only)",
         "point-in-time-correct version of the V3a feature -- the trustworthy number",
         FEATURES_V3B)

['price_ngn', 'region_group', 'month_of_year', 'state_avg_price_pit']

### V4 — encoding comparison: one-hot vs. point-in-time target encoding

The running feature set (V3b's, the best *correct* rung so far) plus the
37-state identity itself, encoded two competing ways:

- **V4a, one-hot**: 37 binary columns (`src/modelling/encoders.one_hot_encode`,
  a `scipy.sparse` matrix, fit on train, aligned onto test).
- **V4b, target encoding**: a single numeric column, each state replaced by
  a smoothed historical mean of the target, computed point-in-time-correctly
  (`src/modelling/encoders.target_encode_pit`, reusing the identical
  expanding-window discipline as V3b).

This rung exists specifically to engage **Ayinla (2023)**, on index-mapped
ordinal encoding of categorical attributes in machine learning. Ayinla's
result argues for a *compact, ordinal, index-mapped* encoding over one-hot's
high-dimensional sparse expansion, on the grounds that ordinal/target-style
encodings let a tree-based learner split on a single informative axis
instead of fragmenting the same signal across dozens of near-duplicate
binary columns. `docs/modelling_notes.md` states plainly whether this
ladder's own result agrees or disagrees with that direction — and, now,
whether the difference between the two encodings even holds its sign across
five repeats, which is the question that decides whether there is a result
to report at all.

The one-hot matrix is built **once**, outside the repeat loop: Task A's data
is identical on every repeat, so rebuilding it per seed would burn time
without changing a single value.

In [15]:
COMMON_V4 = FEATURES_V3B  # carry the correct V3b feature set forward

train_oh, test_oh = one_hot_encode(
    split.train[COMMON_V4 + ["state"]], split.test[COMMON_V4 + ["state"]], col="state"
)
add_rung("V4a", f"+ state identity, ONE-HOT encoded ({train_oh.shape[1] - len(COMMON_V4)} "
                "binary columns)",
         "one-hot encoding of the 37-state categorical, fit on train, aligned onto test",
         kind="onehot")

FEATURES_V4B = COMMON_V4 + ["state_target_enc"]
add_rung("V4b", "+ state identity, POINT-IN-TIME TARGET encoded (1 numeric column)",
         "smoothed, point-in-time-correct target encoding of the 37-state categorical "
         "(smoothing=10); engages Ayinla (2023) -- see docs/modelling_notes.md",
         FEATURES_V4B)

print(f"{len(RUNG_SPECS)} rungs declared: {[s['rung'] for s in RUNG_SPECS]}")
print(f"one-hot matrix: {train_oh.shape[0]:,} x {train_oh.shape[1]} (train)")

7 rungs declared: ['V0', 'V1', 'V2', 'V3a', 'V3b', 'V4a', 'V4b']
one-hot matrix: 925 x 41 (train)


### V5 — external enrichment: not available for this task

**Conditional rung, per the v3 prompt §3.4.** The only external covariate
this platform verifies at daily grain is `fact_weather_daily`, and it is
**NYC-specific** (Open-Meteo's NYC archive, `docs/data_dictionary.md`) — it
carries no Nigerian observations and cannot be joined to a Nigerian state
panel by any honest key. No other externally-sourced, verified covariate
exists in this project for Nigeria. Rather than manufacture a synthetic
feature to fill this slot, **V4b is the final rung of Task A, under both
hyperparameter variants.**

## 8. Running the ladder: every rung, both variants, five times

Seeds 1-5, from `splits.REPEAT_SEEDS`. The seed is passed to
`run_rung(seed=...)`, which overrides `random_state` and nothing else, so
the only thing differing between two repeats of the same rung is LightGBM's
own bagging and column-sampling randomness. Everything else — the data, the
split, the features, the hyperparameters — is byte-identical across repeats.

7 rungs × 2 variants × 5 seeds = 70 fits, plus the two reference predictors
per seed.

In [16]:
VARIANTS = ((VARIANT_ORIGINAL, LGBM_PARAMS),
            (VARIANT_CAPACITY_CONTROLLED, CAPACITY_CONTROLLED_PARAMS))

def run_ladder_for_seed(seed):
    '''Every rung, both hyperparameter variants, at one seed.'''
    out = list(reference_results(seed))
    for spec in RUNG_SPECS:
        if spec["kind"] == "onehot":
            X_train, X_test = train_oh, test_oh
        else:
            X_train = split.train[spec["feature_cols"]]
            X_test = split.test[spec["feature_cols"]]
        for variant, params in VARIANTS:
            out.append(run_rung(
                spec["rung"], spec["description"],
                X_train, y_train, X_test, y_test,
                notes=spec["notes"], params=params, variant=variant, seed=seed,
            ))
    return out

all_results = []
for seed in REPEAT_SEEDS:
    t0 = time.perf_counter()
    seed_results = run_ladder_for_seed(seed)
    all_results.extend(seed_results)
    fitted = [r for r in seed_results if r.variant != VARIANT_REFERENCE]
    print(f"seed {seed}: {len(fitted)} fits in {time.perf_counter() - t0:.1f}s")

print(f"\n{len(all_results)} results total "
      f"({len(RUNG_SPECS)} rungs x 2 variants x {len(REPEAT_SEEDS)} seeds, plus references)")

seed 1: 14 fits in 2.0s


seed 2: 14 fits in 1.9s


seed 3: 14 fits in 2.4s


seed 4: 14 fits in 2.2s


seed 5: 14 fits in 2.0s

80 results total (7 rungs x 2 variants x 5 seeds, plus references)


## 9. Results with uncertainty attached

`mae_ngn` / `mape_pct` keep their original meaning — the **first repeat**
(seed 1) — so the columns written before this pass still say something
concrete. Every claim in `docs/modelling_notes.md` is now made against
`mae_mean` and qualified by `mae_sd`.

Read `mae_sd` first. It is the amount a rung's error moves when nothing
meaningful changes, and any rung-to-rung gap of comparable size is noise.

In [17]:
table_a = aggregate_repeats(all_results, mae_col="mae_ngn")

settings.TABLES_DIR.mkdir(parents=True, exist_ok=True)
table_a_path = settings.TABLES_DIR / "model_ladder_nigeria.csv"
table_a.to_csv(table_a_path, index=False)
print(f"wrote {table_a_path}  ({len(table_a)} rows)")
table_a[["variant", "rung", "mae_ngn", "mae_mean", "mae_sd", "mae_min", "mae_max",
         "mape_mean", "mape_sd", "n_repeats"]]

wrote /app/outputs/tables/model_ladder_nigeria.csv  (16 rows)


,variant,rung,mae_ngn,mae_mean,mae_sd,mae_min,mae_max,mape_mean,mape_sd,n_repeats
0,reference (no model fitted),REF_mean,327.3386,327.3386,0.0000,327.3386,327.3386,22.5437,0.0000,5
1,reference (no model fitted),REF_heuristic,129.3020,129.3020,0.0000,129.3020,129.3020,9.5652,0.0000,5
2,"original (untuned, 300 trees)",V0,159.0962,159.5266,0.6774,158.6183,160.3118,11.4691,0.0457,5
3,capacity-controlled (CV-selected on V0),V0,205.8480,207.4731,1.9277,205.8480,210.6316,14.4105,0.1212,5
4,"original (untuned, 300 trees)",V1,159.0962,159.5266,0.6774,158.6183,160.3118,11.4691,0.0457,5
5,capacity-controlled (CV-selected on V0),V1,205.8480,207.4731,1.9277,205.8480,210.6316,14.4105,0.1212,5
6,"original (untuned, 300 trees)",V2,184.2418,200.8836,14.4423,184.2418,212.9436,14.0166,0.7421,5
7,capacity-controlled (CV-selected on V0),V2,289.6359,244.1159,26.1775,228.5897,289.6359,16.8434,1.7559,5
8,"original (untuned, 300 trees)",V3a,215.2381,217.5221,2.7132,214.8367,220.5684,15.1275,0.2175,5
9,capacity-controlled (CV-selected on V0),V3a,242.5188,249.3631,13.5852,235.3644,271.5814,17.1703,0.8983,5


## 10. Paired comparisons: which differences survive repetition?

Every rung ran on the same five seeds, so the honest comparison subtracts
**within** a seed and summarises those five differences, rather than
comparing two marginal distributions and eyeballing whether the error bars
overlap. `tests/test_repeats.py` asserts this pairing behaves correctly.

`mean_diff_mae` is rung B minus rung A: **negative means B is better.**
`n_same_direction` counts how many of the five repeats agreed with the
mean's sign — the number that decides whether a difference is reportable.
Nothing here is a significance test, and none is claimed: with five repeats
these are descriptive indications of stability, and a difference that does
not hold its sign in all five repeats is not an established effect.

In [18]:
RUNG_ORDER = [s["rung"] for s in RUNG_SPECS]

best_original = best_rung_by_mean_mae(table_a, VARIANT_ORIGINAL)
best_capacity = best_rung_by_mean_mae(table_a, VARIANT_CAPACITY_CONTROLLED)
print(f"best rung by mean MAE -- original: {best_original}, capacity-controlled: {best_capacity}")
print("(V3a and the REF_* rows are excluded from 'best': leaky, and not rungs)")

# The comparisons that matter, on top of every adjacent step. "V0 vs the best
# rung" is only a comparison when some rung actually beat V0; when V0 IS the
# best rung, a V0-vs-V0 row would be a meaningless tie dressed up as a result,
# so it is reported as the finding it is instead.
special = [
    Pair("V3a", "V3b", label="leaky vs point-in-time correct"),
    Pair("V4a", "V4b", label="one-hot vs point-in-time target encoding"),
]
for variant, best in ((VARIANT_ORIGINAL, best_original),
                      (VARIANT_CAPACITY_CONTROLLED, best_capacity)):
    if best == "V0":
        print(f"  [{variant}] the best rung IS V0: no engineered rung beat the naive baseline.")
    else:
        special.append(Pair("V0", best, label="V0 vs best rung"))

# Deduplicate: V3a->V3b and V4a->V4b are also adjacent steps, and one row per
# comparison is clearer than the same numbers under two labels.
seen = {(p.rung_a, p.rung_b) for p in special}
pairs = special + [p for p in adjacent_pairs(RUNG_ORDER) if (p.rung_a, p.rung_b) not in seen]

paired_a = paired_comparisons(all_results, pairs, task="nigeria", mae_col="mae_ngn")
paired_a_path = settings.TABLES_DIR / "ladder_paired_comparisons_nigeria.csv"
paired_a.to_csv(paired_a_path, index=False)
print(f"wrote {paired_a_path}  ({len(paired_a)} comparisons)")

paired_a[["variant", "rung_a", "rung_b", "comparison", "mean_diff_mae", "sd_diff_mae",
          "mean_over_sd", "n_same_direction", "direction_consistent", "better_rung"]]

best rung by mean MAE -- original: V0, capacity-controlled: V0
(V3a and the REF_* rows are excluded from 'best': leaky, and not rungs)
  [original (untuned, 300 trees)] the best rung IS V0: no engineered rung beat the naive baseline.
  [capacity-controlled (CV-selected on V0)] the best rung IS V0: no engineered rung beat the naive baseline.


wrote /app/outputs/tables/ladder_paired_comparisons_nigeria.csv  (12 comparisons)


,variant,rung_a,rung_b,comparison,mean_diff_mae,sd_diff_mae,mean_over_sd,n_same_direction,direction_consistent,better_rung
0,"original (untuned, 300 trees)",V3a,V3b,leaky vs point-in-time correct,-10.7379,2.4060,-4.463,5,True,V3b
1,"original (untuned, 300 trees)",V4a,V4b,one-hot vs point-in-time target encoding,-0.3290,3.7761,-0.087,3,False,V4b
2,"original (untuned, 300 trees)",V0,V1,adjacent,0.0000,0.0000,NaN,0,False,tie
3,"original (untuned, 300 trees)",V1,V2,adjacent,41.3570,13.8310,2.990,5,True,V1
4,"original (untuned, 300 trees)",V2,V3a,adjacent,16.6385,14.9328,1.114,5,True,V2
5,"original (untuned, 300 trees)",V3b,V4a,adjacent,5.6453,2.7160,2.079,5,True,V3b
6,capacity-controlled (CV-selected on V0),V3a,V3b,leaky vs point-in-time correct,-16.5813,14.6159,-1.134,5,True,V3b
7,capacity-controlled (CV-selected on V0),V4a,V4b,one-hot vs point-in-time target encoding,-0.8268,10.9228,-0.076,3,False,V4b
8,capacity-controlled (CV-selected on V0),V0,V1,adjacent,0.0000,0.0000,NaN,0,False,tie
9,capacity-controlled (CV-selected on V0),V1,V2,adjacent,36.6428,27.1984,1.347,5,True,V1


### Do the rungs beat the reference predictors at all?

The references fit no model and have no seed, so a paired difference against
them is not a like-for-like pairing — what matters is simply whether each
rung's error clears theirs, and on how many of the five repeats. Counted
directly below, per variant, for every rung.

In [19]:
ref_rows = table_a[table_a["variant"] == VARIANT_REFERENCE].set_index("rung")
ref_walk_mae = float(ref_rows.loc["REF_heuristic", "mae_mean"])
ref_mean_mae = float(ref_rows.loc["REF_mean", "mae_mean"])
print(f"REF_heuristic (random walk): MAE=₦{ref_walk_mae:,.2f}")
print(f"REF_mean (training mean):    MAE=₦{ref_mean_mae:,.2f}")
print()

from src.modelling.repeats import results_to_long_frame
long_a = results_to_long_frame(all_results, mae_col="mae_ngn")
fitted_long = long_a[long_a["variant"] != VARIANT_REFERENCE]

for variant in (VARIANT_ORIGINAL, VARIANT_CAPACITY_CONTROLLED):
    print(f"[{variant}]")
    scoped = fitted_long[fitted_long["variant"] == variant]
    for rung in RUNG_ORDER:
        maes = scoped[scoped["rung"] == rung]["mae_ngn"]
        beat_walk = int((maes < ref_walk_mae).sum())
        beat_mean = int((maes < ref_mean_mae).sum())
        print(f"  {rung:4s} beats random walk in {beat_walk}/{len(maes)} repeats;  "
              f"beats training mean in {beat_mean}/{len(maes)} repeats")
    print()

REF_heuristic (random walk): MAE=₦129.30
REF_mean (training mean):    MAE=₦327.34

[original (untuned, 300 trees)]
  V0   beats random walk in 0/5 repeats;  beats training mean in 5/5 repeats
  V1   beats random walk in 0/5 repeats;  beats training mean in 5/5 repeats
  V2   beats random walk in 0/5 repeats;  beats training mean in 5/5 repeats
  V3a  beats random walk in 0/5 repeats;  beats training mean in 5/5 repeats
  V3b  beats random walk in 0/5 repeats;  beats training mean in 5/5 repeats
  V4a  beats random walk in 0/5 repeats;  beats training mean in 5/5 repeats
  V4b  beats random walk in 0/5 repeats;  beats training mean in 5/5 repeats

[capacity-controlled (CV-selected on V0)]
  V0   beats random walk in 0/5 repeats;  beats training mean in 5/5 repeats
  V1   beats random walk in 0/5 repeats;  beats training mean in 5/5 repeats
  V2   beats random walk in 0/5 repeats;  beats training mean in 5/5 repeats
  V3a  beats random walk in 0/5 repeats;  beats training mean in 5/5 rep

In [20]:
inconsistent = paired_a[~paired_a["direction_consistent"]]
consistent = paired_a[paired_a["direction_consistent"]]
print(f"{len(consistent)} of {len(paired_a)} comparisons held their direction across all "
      f"{len(REPEAT_SEEDS)} repeats.")
print()
if len(inconsistent):
    print("NOT consistent in direction -- these must not be reported as established effects:")
    for _, row in inconsistent.iterrows():
        print(f"  [{row['variant'][:22]:22s}] {row['rung_a']:>4s} -> {row['rung_b']:<4s} "
              f"mean {row['mean_diff_mae']:+9.2f}  sd {row['sd_diff_mae']:8.2f}  "
              f"{row['n_same_direction']}/{row['n_repeats']} agree")
else:
    print("every comparison held its direction across all repeats.")

6 of 12 comparisons held their direction across all 5 repeats.

NOT consistent in direction -- these must not be reported as established effects:
  [original (untuned, 300]  V4a -> V4b  mean     -0.33  sd     3.78  3/5 agree
  [original (untuned, 300]   V0 -> V1   mean     +0.00  sd     0.00  0/5 agree
  [capacity-controlled (C]  V4a -> V4b  mean     -0.83  sd    10.92  3/5 agree
  [capacity-controlled (C]   V0 -> V1   mean     +0.00  sd     0.00  0/5 agree
  [capacity-controlled (C]   V2 -> V3a  mean     +5.25  sd    30.25  4/5 agree
  [capacity-controlled (C]  V3b -> V4a  mean     +4.33  sd     4.60  4/5 agree


## 11. The figure: rungs with error bars, against the reference predictors

In [21]:
vizstyle.apply_style()

summary = table_a.set_index(["variant", "rung"])
rungs = RUNG_ORDER
mape_o = [summary.loc[(VARIANT_ORIGINAL, r), "mape_mean"] for r in rungs]
sd_o = [summary.loc[(VARIANT_ORIGINAL, r), "mape_sd"] for r in rungs]
mape_c = [summary.loc[(VARIANT_CAPACITY_CONTROLLED, r), "mape_mean"] for r in rungs]
sd_c = [summary.loc[(VARIANT_CAPACITY_CONTROLLED, r), "mape_sd"] for r in rungs]
ref_mean_mape = summary.loc[(VARIANT_REFERENCE, "REF_mean"), "mape_mean"]
ref_walk_mape = summary.loc[(VARIANT_REFERENCE, "REF_heuristic"), "mape_mean"]

fig, ax = plt.subplots(figsize=(8.6, 5.6))
x = np.arange(len(rungs))
width = 0.38

# Colour marks the VARIANT, hatching marks the leaky rung. Colouring V3a
# separately would have made the two variants indistinguishable at exactly
# the rung where their difference is worth seeing.
colours_o = [vizstyle.PALETTE[0]] * len(rungs)
colours_c = [vizstyle.PALETTE[2]] * len(rungs)
hatches = ["////" if r == "V3a" else None for r in rungs]

ax.bar(x - width / 2, mape_o, width=width, yerr=sd_o, capsize=3,
       color=colours_o, hatch=hatches, edgecolor="white", linewidth=0.6,
       error_kw={"ecolor": vizstyle.TEXT_COLOUR, "elinewidth": 1.0},
       label="original (untuned, 300 trees)")
ax.bar(x + width / 2, mape_c, width=width, yerr=sd_c, capsize=3,
       color=colours_c, hatch=hatches, edgecolor="white", linewidth=0.6,
       error_kw={"ecolor": vizstyle.TEXT_COLOUR, "elinewidth": 1.0},
       label="capacity-controlled (CV-selected on V0)")

ax.axhline(ref_walk_mape, color=vizstyle.PALETTE[3], linestyle="--", linewidth=1.4,
           label=f"REF random walk ({ref_walk_mape:.1f}%)")
ax.axhline(ref_mean_mape, color=vizstyle.MUTED_COLOUR, linestyle=":", linewidth=1.4,
           label=f"REF training mean ({ref_mean_mape:.1f}%)")

top = max(max(mape_o), max(mape_c), ref_mean_mape)
ax.set_ylabel("MAPE on held-out test set (%)", fontsize=10)
ax.set_xlabel("Ladder rung", fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(rungs, fontsize=10)
ax.tick_params(axis="y", labelsize=9)
ax.set_title("Nigerian petrol price, one month ahead: MAPE by rung, with repeat spread",
             fontsize=11, fontweight="bold")
ax.set_ylim(0, top * 1.30)
ax.legend(fontsize=8.5, loc="upper left", ncol=2)
fig.tight_layout()

fig_path = vizstyle.finish(
    fig, settings.FIGURES_DIR / "fig_model_ladder_nigeria.png",
    "Source: this platform's own gold layer (fact_fuel_price_monthly)",
    f"Bars are the mean of {len(REPEAT_SEEDS)} repeats (seeds {REPEAT_SEEDS[0]}-{REPEAT_SEEDS[-1]}, "
    "varying the model's random_state only; Task A does no sampling); error bars are one standard "
    "deviation across those repeats. Dashed and dotted lines are reference predictors that fit no "
    "model: a random walk, and the training mean. V3a (hatched) is computed with future months "
    "present and is reported only for comparison against V3b. Which rung-to-rung differences hold "
    "their direction across all repeats is reported in outputs/tables/"
    "ladder_paired_comparisons.csv, not readable from bar heights alone.",
)
print(f"wrote {fig_path}")

2026-09-19 09:23:43 | INFO    | src.viz.style                | figure written: fig_model_ladder_nigeria.png


wrote /app/outputs/figures/fig_model_ladder_nigeria.png


## 12. Notebook runtime

In [22]:
elapsed = time.perf_counter() - NOTEBOOK_STARTED
print(f"notebook 01 wall-clock time: {elapsed:.1f}s ({elapsed/60:.2f} min)")
print(f"  {len(RUNG_SPECS)} rungs x 2 variants x {len(REPEAT_SEEDS)} repeats = "
      f"{len(RUNG_SPECS) * 2 * len(REPEAT_SEEDS)} fits, plus a {tuning.seconds:.1f}s CV search")
con.close()

notebook 01 wall-clock time: 14.0s (0.23 min)
  7 rungs x 2 variants x 5 repeats = 70 fits, plus a 1.4s CV search
